In [1]:
import os
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

In [2]:
class WordDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {
            cls: idx for idx, cls in enumerate(self.classes)
        }

        for cls in self.classes:
            class_dir = os.path.join(root_dir, cls)

            if not os.path.isdir(class_dir):
                continue

            for file in os.listdir(class_dir):
                if file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.samples.append(
                        (
                            os.path.join(class_dir, file),
                            self.class_to_idx[cls]
                        )
                    )

        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((64, 128)),

            transforms.RandomRotation(5),
            transforms.RandomAffine(
                degrees=0,
                translate=(0.05, 0.05)
            ),

            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("L")
        image = self.transform(image)

        return image, label

In [3]:
dataset = WordDataset("custom")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [4]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(128 * 8 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SmallCNN(num_classes=len(dataset.classes)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_val_acc = 0

for epoch in range(25):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = 100 * correct / total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Loss: {train_loss / len(train_loader):.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(model.state_dict(), "best_word_model.pth")

        torch.save(dataset.classes, "classes.pth")

        print("Saved best model")

Epoch 1/20 | Train Loss: 3.4130 | Val Acc: 0.00%
Epoch 2/20 | Train Loss: 3.3785 | Val Acc: 9.86%
Saved best model
Epoch 3/20 | Train Loss: 3.1652 | Val Acc: 21.13%
Saved best model
Epoch 4/20 | Train Loss: 2.7642 | Val Acc: 18.31%
Epoch 5/20 | Train Loss: 2.4164 | Val Acc: 29.58%
Saved best model
Epoch 6/20 | Train Loss: 2.1490 | Val Acc: 42.25%
Saved best model
Epoch 7/20 | Train Loss: 1.6393 | Val Acc: 53.52%
Saved best model
Epoch 8/20 | Train Loss: 1.3377 | Val Acc: 63.38%
Saved best model
Epoch 9/20 | Train Loss: 0.9945 | Val Acc: 66.20%
Saved best model
Epoch 10/20 | Train Loss: 0.8586 | Val Acc: 77.46%
Saved best model
Epoch 11/20 | Train Loss: 0.7120 | Val Acc: 77.46%
Epoch 12/20 | Train Loss: 0.5430 | Val Acc: 87.32%
Saved best model
Epoch 13/20 | Train Loss: 0.4727 | Val Acc: 83.10%
Epoch 14/20 | Train Loss: 0.4330 | Val Acc: 84.51%
Epoch 15/20 | Train Loss: 0.3694 | Val Acc: 74.65%
Epoch 16/20 | Train Loss: 0.3026 | Val Acc: 85.92%
Epoch 17/20 | Train Loss: 0.3059 | Val Acc